In [1]:
pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 MB 5.3 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 10.7 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.5-py2.py3-none-any.whl size=317747860 sha256=76681f80fbc7ffd3b8ac144900ff318117dd003fac0fd62aec54e93de349c145
  Stored in directory: /home/jovyan/.cache/pip/wheels/0c/7f/b4/0e68c6d8d89d2e582e5498ad88616c16d7c19028680e9d3840
Successfully built pyspark
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, size, length, lower, when
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import Row

import json
import os


spark = SparkSession.builder \
    .appName("Predict Internship vs Full-Time") \
    .config("spark.executor.memory", "6g") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()


hdfs_path = "hdfs://namenode:8020/user/hadoop/csv_datasets/"

job_skills_df = spark.read.option("header", True).csv(hdfs_path + "job_skills.csv")
linkedin_jobs_df = spark.read.option("header", True).csv(hdfs_path + "linkedin_job_postings.csv")
job_summary_df = spark.read.option("header", True).csv(hdfs_path + "job_summary.csv")


job_skills_df = job_skills_df.dropna(subset=["job_link", "job_skills"])
linkedin_jobs_df = linkedin_jobs_df.dropna(subset=["job_link", "job_title"])
job_summary_df = job_summary_df.dropna(subset=["job_link", "job_summary"])

# Join datasets
data = job_skills_df.join(
    linkedin_jobs_df.select("job_link", "job_title"),
    on="job_link",
    how="inner"
).join(
    job_summary_df.select("job_link", "job_summary"),
    on="job_link",
    how="inner"
)
# If "intern" keyword appears in job title ➔ label 1
# Else ➔ label 0
data = data.withColumn(
    "label",
    when(
        lower(col("job_title")).contains("intern"),
        1
    ).otherwise(0)
)



# Skill count
data = data.withColumn("skill_count", size(split(col("job_skills"), ",")))

# Summary length
data = data.withColumn("summary_length", length(col("job_summary")))


assembler = VectorAssembler(
    inputCols=["skill_count", "summary_length"],
    outputCol="features"
)


classifier = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)


pipeline = Pipeline(stages=[assembler, classifier])


train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

paramGrid = ParamGridBuilder() \
    .addGrid(classifier.regParam, [0.0, 0.1]) \
    .addGrid(classifier.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=BinaryClassificationEvaluator(labelCol="label"),
    numFolds=5
)

cvModel = crossval.fit(train_data)

predictions = cvModel.transform(test_data)

evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)
auc_score = evaluator_auc.evaluate(predictions)

print(f"AUC Score on Test Data: {auc_score}")


AUC Score on Test Data: 0.5420174387857917


In [2]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import Row

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = accuracy_evaluator.evaluate(predictions)
print(f"Accuracy on Test Data: {accuracy}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
f1_score = f1_evaluator.evaluate(predictions)
print(f"F1 Score on Test Data: {f1_score}")

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
)
precision = precision_evaluator.evaluate(predictions)
print(f"Precision on Test Data: {precision}")

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall"
)
recall = recall_evaluator.evaluate(predictions)
print(f"Recall on Test Data: {recall}")


Accuracy on Test Data: 0.9919412310809613
F1 Score on Test Data: 0.9879281482460777
Precision on Test Data: 0.9839474059184131
Recall on Test Data: 0.9919412310809613


In [ ]:
model.bestModel.save("hdfs://namenode:8020/user/hadoop/csv_files/Machine_Learning/models/senior_level_predictions")
metrics_data = [
    Row(metric="Accuracy", value=accuracy),
    Row(metric="F1 Score", value=f1_score),
    Row(metric="Precision", value=precision),
    Row(metric="Recall", value=recall)
]

metrics_df = spark.createDataFrame(metrics_data)

hdfs_path = "hdfs://namenode:8020/user/hadoop/csv_files/Machine_Learning/outputs/senior_level_prediction/senior_level_predictions.csv"

metrics_df.write.option("header", "true").csv(hdfs_path)
